# Chunking Strategies: The Shape Matters

## Text Chunking Strategy Comparison

| Method             | Strength                      | Trade-off                         | Best For                             |
| :----------------- | :---------------------------- | :-------------------------------- | :----------------------------------- |
| **Fixed-Size**     | Simple, predictable chunks    | Ignores structure, breaks meaning | Raw or unstructured text             |
| **Sentence**       | Preserves complete thoughts   | Inconsistent sizes                | RAG, Q&A systems                     |
| **Paragraph**      | Aligns with semantic units    | Large variance in length          | Docs, manuals, instructional content |
| **Sliding Window** | Maintains full context        | Redundant, compute-heavy          | Reranking, high-recall retrieval     |
| **Recursive**      | Flexible, handles messy input | Heuristic, sometimes brittle      | Scraped web content, mixed sources   |
| **Semantic**       | High-quality, meaning-aware   | Slower, resource-intensive        | Legal, research, critical QA         |


### 1. Fixed-Size Chunking
The Approach: Define a number of tokens per chunk (e.g., 200) with a small overlap buffer to preserve context.
Pros:
- Simple to implement
- Consistent chunk sizes
- Predictable processing

Cons:
- Ignores natural language boundaries
- May split mid-sentence or mid-thought
- No semantic awareness

Best for: Documents lacking consistent formatting, initial prototyping

In [12]:
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [13]:
text = "The HNSW algorithm builds a multi-layer graph where each node represents a vector. The algorithm starts by inserting vectors into the bottom layer and then selectively promotes some to higher layers based on probability. This creates shortcuts that allow for faster traversal during search operations."
fixed_size_chunk(text)

['The HNSW algorithm builds a multi-layer graph where each node represents a vector. The algorithm starts by inserting vectors into the bottom layer and then selectively promotes some to higher layers based on probability. This creates shortcuts that allow for faster traversal during search operations.']

### 2. Sentence-Based Chunking
The Approach: Break documents into sentences using a tokenizer, then group sentences into chunks under a specified word count.
Pros:
- Preserves complete thoughts
- Natural language boundaries
- Good semantic coherence

Cons:
- Irregular chunk lengths
- Sentence size varies significantly
- May not respect topic boundaries

Best for: RAG systems, Q&A applications, general text processing

In [14]:
# ! pip install nltk
from nltk.tokenize import sent_tokenize


def sentence_chunk(text, max_words=5):
    sentences = sent_tokenize(text)
    chunks, buffer, length = [], [], 0

    for sent in sentences:
        count = len(sent.split())
        if length + count > max_words:
            chunks.append(" ".join(buffer))
            buffer, length = [], 0
        buffer.append(sent)
        length += count

    if buffer:
        chunks.append(" ".join(buffer))
    return chunks

In [15]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/humengqing/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/humengqing/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
text= "Hello world. This is Qdrant. Hello world. This is Qdrant."
print(sentence_chunk(text))

['Hello world. This is Qdrant.', 'Hello world. This is Qdrant.']


### 3. Paragraph-Based Chunking
The Approach: Split on paragraph breaks, leveraging existing document structure.
Pros:
- Aligns with natural topic boundaries
- Semantically rich by default
- Respects author’s organization

Cons:
- Unpredictable sizes (single line to whole page)
- May need token limits or fallback splitting
- Depends on clean document structure

Best for: Articles, blogs, documentation, books, emails

In [17]:
def paragraph_chunk(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]

### 4. Sliding Window Chunking
The Approach: Create overlapping chunks to maintain context continuity.
Pros:
- Maintains context at boundaries
- Higher recall potential
- Reduces information loss

Cons:
- Storage redundancy (typically 20-50% overhead)
- Increased processing costs
- May return duplicate information

Best for: Critical applications where missing information is costly, reranking systems

In [18]:
def sliding_window(text, window=200, stride=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words) - window + 1, stride):
        chunk = " ".join(words[i:i + window])
        chunks.append(chunk)
    return chunks

### 5. Recursive Chunking
The Approach: Use a fallback hierarchy of separators when data doesn’t follow predictable structure.

Pros:
- Adapts to messy or inconsistent input
- Preserves semantic coherence when possible
- Handles various document formats

Cons:
- Heuristic-based, results may be inconsistent
- Complex logic
- May not work perfectly with all content types
Best for: Scraped web content, mixed formats, CMS exports

In [19]:
# ! pip install langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""]
)

text = """
Hello, world! More text here. another line.
Hello, world! More text here. another line......
"""
chunks = splitter.split_text(text)

### 6. Semantic-Aware Chunking
The Approach: Use embeddings to detect meaning shifts and break at topic boundaries.

Pros:
- High semantic precision
- Each chunk carries coherent ideas
- Optimal for complex documents

Cons:
- Computationally expensive (requires embedding entire document)
- Requires additional model inference
- Slower processing pipeline

Best for: Legal documents, research papers, critical applications requiring high precision

In [20]:
from sentence_transformers import SentenceTransformer
import numpy as np

def semantic_chunking(text, similarity_threshold=0.5):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = text.split('.')
    embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        # Calculate cosine similarity between consecutive sentences
        similarity = np.dot(embeddings[i-1], embeddings[i]) / (
            np.linalg.norm(embeddings[i-1]) * np.linalg.norm(embeddings[i])
        )
        
        if similarity < similarity_threshold:
            chunks.append('. '.join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])
    
    chunks.append('. '.join(current_chunk))
    return chunks

# Adding Meaning with Metadata


In [21]:
{
  "document_id": "collection-config-guide",
  "document_title": "What is a Vector Database",
  "section_title": "What Is a Vector",
  "chunk_index": 7,
  "chunk_count": 15,
  "url": "https://qdrant.tech/documentation/manage-data/collections/",
  "tags": ["qdrant", "vector search", "point", "vector", "payload"],
  "source_type": "documentation", 
  "created_at": "2025-01-15T10:00:00Z",
  "content": "There are three key elements that define a vector in vector search: the ID, the dimensions, and the payload. These components work together to represent a vector effectively within the system...",
  "word_count": 45,
  "char_count": 287
}

{'document_id': 'collection-config-guide',
 'document_title': 'What is a Vector Database',
 'section_title': 'What Is a Vector',
 'chunk_index': 7,
 'chunk_count': 15,
 'url': 'https://qdrant.tech/documentation/manage-data/collections/',
 'tags': ['qdrant', 'vector search', 'point', 'vector', 'payload'],
 'source_type': 'documentation',
 'created_at': '2025-01-15T10:00:00Z',
 'content': 'There are three key elements that define a vector in vector search: the ID, the dimensions, and the payload. These components work together to represent a vector effectively within the system...',
 'word_count': 45,
 'char_count': 287}

In [22]:
from qdrant_client import models

# Only show results from a specific article
filter = models.Filter(
    must=[
        models.FieldCondition(
            key="document_id", match=models.MatchValue(value="collection-config-guide")
        )
    ]
)

In [23]:
# Find vectors that also contain the keyword "HNSW" in their content
filter = models.Filter(
    must=[
        models.FieldCondition(
            key="content", # The field with the full-text index
            match=models.MatchText(text="HNSW algorithm")
        )
    ]
)

In [24]:
# Top result per document - get the most relevant chunk from each source
group_by = "document_id"

In [25]:
# Filter by user permissions
filter = models.Filter(
    must=[
        models.FieldCondition(
            key="access_level", match=models.MatchValue(value="public")
        )
    ]
)

In [26]:
def search_with_filters(query, document_type=None, date_range=None):
    """Search with metadata filtering"""

    # Build filter conditions
    filter_conditions = []

    if document_type:
        filter_conditions.append(
            models.FieldCondition(
                key="source_type", match=models.MatchValue(value=document_type)
            )
        )

    if date_range:
        filter_conditions.append(
            models.FieldCondition(
                key="created_at",
                range=models.Range(gte=date_range["start"], lte=date_range["end"]),
            )
        )

    # Execute search
    query_filter = models.Filter(must=filter_conditions) if filter_conditions else None

    results = client.query_points(
        collection_name="documents",
        query=generate_embedding(query),
        query_filter=query_filter,
        limit=5,
    )

    return results